In [ ]:
from enum import Enum, auto

In [ ]:
class System_State(Enum):
    START = auto()
    NORMAL = auto()
    RETRY = auto()
    HUMAN = auto()
    END = auto()
    JSON_ERROR = auto()
    RETRY_ERROR = auto()
    TOOL_ERROR = auto()

In [ ]:
for item in System_State:
    print(item.value)

In [ ]:
from domain.graph import BuilderGraph


In [ ]:
g = BuilderGraph(
        graph_id="content_creation_v1",
        name="内容创作流程",
        description="从需求分析到多平台发布的完整内容创作业务图，支持自动评判和人工审核双轨",
    ).build_content_creation_graph()


In [ ]:
g2 = BuilderGraph(
        graph_id="critic_ai_graph",
        name="内容AI审核流程",
        description="自动评判流程",
    ).build_critic_ai_graph()

In [ ]:
BuilderGraph.visualize_pyvis(g)

In [ ]:
BuilderGraph.visualize_pyvis(g2,output_path="./graph_visualization2.html")

In [ ]:
list(g._nodes.keys())

In [ ]:
list(g._edges.keys())

In [ ]:
from domain.event import ToolEventFactory
from infra.eventbus import EventBus
from domain.agent.write.tools import *
from infra.tool.tools_attach_methods import *

In [ ]:
bus = EventBus()


In [ ]:
factory2 = ToolEventFactory(prefix="infra")

In [ ]:
factory2.tool("read_files").emit_called()

In [ ]:
factory = ToolEventFactory(prefix="infra").export_class("./domain/tools/static_tools.py")._resigister_bus(bus)

In [ ]:
bus._handlers

In [ ]:
factory = ToolEventFactory(prefix="infra").export_class("./domain/tools/static_tools.py")

In [ ]:
factory2 = ToolEventFactory(prefix="infra")._build()

In [1]:
"""
Writer Agent 运行示例
演示如何使用 WriteAgent 进行小说创作
"""

import asyncio
import os
from dotenv import load_dotenv

# 加载环境变量
load_dotenv()

from domain.agent.write.writeAgent import WriteAgent
from infra.LLM.LLM_infra import LLM_Client, LLM_Model_Provider


async def run_writer_agent_example():
    """
    Writer Agent 运行示例
    
    这个示例展示了：
    1. 初始化 LLM 客户端
    2. 创建 WriteAgent 实例
    3. 启动事件总线监听工具调用
    4. 提交创作任务并观察执行过程
    """
    
    print("=" * 60)
    print("Writer Agent 运行示例")
    print("=" * 60)
    
    # 1. 初始化 LLM 客户端
    print("\n[1] 初始化 LLM 客户端...")
    llm_client = LLM_Client(
        url=os.getenv("LLM_BASE_URL", "https://open.bigmodel.cn/api/paas/v4/"),
        model_class="MiniMax-M2.7",
        model_provider=LLM_Model_Provider.MINMAX,
        max_tokens=131072,
    )
    print("✓ LLM 客户端初始化完成")
    
    # 2. 创建 WriteAgent 实例
    print("\n[2] 创建 WriteAgent 实例...")
    writer_agent = WriteAgent(
        id="writer_001",
        name="NovelWriter",
        llm=llm_client
    )
    print(f"✓ Agent ID: {writer_agent.id}")
    print(f"✓ Agent Name: {writer_agent.name}")
    print(f"✓ 可用工具: {writer_agent.avilable_tools}")
    
    
    # 3. 定义创作任务
    print("\n[3] 定义创作任务...")
    creative_prompt = """
请帮我写一篇短篇小说，要求如下：

题材：科幻悬疑
风格：紧张刺激，带有哲学思考
主要角色：一位人工智能研究员发现了一个神秘的信号
情节要求：
- 开头要引人入胜
- 中间要有转折
- 结尾要有深意
字数：约30000字
目标读者：成年科幻爱好者
特殊要求：避免过于技术化的术语，保持故事的可读性
"""
    
    print(f"创作任务:\n{creative_prompt}")
    
    # 4. 启动 Agent
    print("\n[5] 启动 Writer Agent...")
    print("=" * 60)
    
    try:
        await writer_agent.start(prompt=creative_prompt)
        
        print("\n" + "=" * 60)
        print("Agent 执行完成！")
        print("=" * 60)
        
        # 6. 查看最终状态
        print("\n[6] 最终状态:")
        final_state = writer_agent.states_manage.get_state()
        print(f"   是否完成: {final_state.get('is_finished', False)}")
        print(f"   完成原因: {final_state.get('finish_reason', 'N/A')}")
        print(f"   思考过程: {final_state.get('think', 'N/A')[:300]}...")
        print(f"   工具调用历史: {' -> '.join(final_state.get('tool_history', []))}")
        print(f"   重试次数: {final_state.get('retry', 0)}")
        
        # 7. 查看工具响应
        if final_state.get('tool_respond'):
            print(f"\n[7] 工具响应摘要:")
            for i, respond in enumerate(final_state['tool_respond'], 1):
                preview = str(respond)[:150] + "..." if len(str(respond)) > 150 else respond
                print(f"   {i}. {preview}")
        
        print("\n✓ 示例执行完成！")
        return writer_agent
        
    except Exception as e:
        print(f"\n❌ 执行出错: {str(e)}")
        import traceback
        traceback.print_exc()


async def run_simple_example():
    """
    简化版示例 - 快速测试
    """
    print("=" * 60)
    print("Writer Agent 简化示例")
    print("=" * 60)
    
    # 初始化
    llm_client = LLM_Client(
        url=os.getenv("LLM_BASE_URL", "https://open.bigmodel.cn/api/paas/v4/"),
        model_class="glm-5.1",
        model_provider=LLM_Model_Provider.GLM,
        max_tokens=131072,
    )
    
    writer_agent = WriteAgent(
        id="writer_simple",
        name="SimpleWriter",
        llm=llm_client
    )
    
    # 简单任务
    prompt = "写一个关于时间旅行的短篇故事，50000字左右"
    
    print(f"\n任务: {prompt}\n")
    print("开始执行...\n")
    
    await writer_agent.start(prompt=prompt)
    
    # 输出结果
    state = writer_agent.states_manage.get_state()
    print("\n执行结果:")
    print(f"完成状态: {state.get('is_finished')}")
    print(f"工具调用: {' -> '.join(state.get('tool_history', []))}")



In [2]:
from domain.event import ToolEventFactory
from infra.eventbus import EventBus
from domain.agent.write.tools import *
from infra.tool.tools_attach_methods import *



In [3]:
bus = EventBus()
factory = ToolEventFactory(prefix="infra")

In [4]:
def run_async(coro):
    try:
        loop = asyncio.get_running_loop()
    except RuntimeError:
        return asyncio.run(coro)

    # 已在事件循环中
    return loop.create_task(coro)


In [ ]:
agent = await run_writer_agent_example()

Writer Agent 运行示例

[1] 初始化 LLM 客户端...
✓ LLM 客户端初始化完成

[2] 创建 WriteAgent 实例...
✓ Agent ID: writer_001
✓ Agent Name: NovelWriter
✓ 可用工具: ['system', 'memory', 'write_agent']

[3] 定义创作任务...
创作任务:

请帮我写一篇短篇小说，要求如下：

题材：科幻悬疑
风格：紧张刺激，带有哲学思考
主要角色：一位人工智能研究员发现了一个神秘的信号
情节要求：
- 开头要引人入胜
- 中间要有转折
- 结尾要有深意
字数：约30000字
目标读者：成年科幻爱好者
特殊要求：避免过于技术化的术语，保持故事的可读性


[5] 启动 Writer Agent...
🔥 start 设置 prompt: 
请帮我写一篇短篇小说，要求如下：

题材：科幻悬疑
风格：紧张刺激，带有哲学思考
主要角色：一位人工智能研究员发现了一个神秘的信号
情节要求：
- 开头要引人入胜
- 中间要有转折
- 结尾要有深意
字数：约30000字
目标读者：成年科幻爱好者
特殊要求：避免过于技术化的术语，保持故事的可读性


请帮我写一篇短篇小说，要求如下：

题材：科幻悬疑
风格：紧张刺激，带有哲学思考
主要角色：一位人工智能研究员发现了一个神秘的信号
情节要求：
- 开头要引人入胜
- 中间要有转折
- 结尾要有深意
字数：约30000字
目标读者：成年科幻爱好者
特殊要求：避免过于技术化的术语，保持故事的可读性

{
  "think": "用户需要一篇约30000字的科幻悬疑短篇小说。这是一个复杂的创作任务，需要先进行系统性的规划。根据工作原则，对于这种长篇创作，我应该先进行需求分析，然后生成大纲，最后进行写作。由于篇幅较长（30000字），我会将这个任务分解为几个阶段：首先分析需求，然后生成详细的章节大纲，最后分章节完成写作。",
  "tool_calls": [
    {
      "tool_name": "requirements_analysis",
      "arguments": {
        "user_prompt": "请帮我写一篇短篇小说，要求如下：\n\n题材：科幻悬疑\n风格：紧张刺

In [6]:
agent.states["tool_respond_full"]

AttributeError: 'NoneType' object has no attribute 'states'

In [ ]:
asyncio.run(run_simple_example())
   

In [ ]:
        # responds = s.get("tool_respond", [])
        # if responds:
        #     parts.append(f"- 工具反馈（共 {len(responds)} 条）：")

        #     for i, item in enumerate(responds):
        #         tool_name = item["tool_name"]
        #         respond   = item["respond"]
        #         is_recent = i >= len(responds) - RECENT_COUNT

        #         if is_recent:
        #             # 最近几条：超长则截断并提示可查询
        #             if len(respond) > MAX_INLINE_LEN:
        #                 parts.append(
        #                     f"  [{tool_name}] {respond[:MAX_INLINE_LEN]}... "
        #                     f"(已截断，可调用 query_tool_respond 获取完整内容)"
        #                 )
        #             else:
        #                 parts.append(f"  [{tool_name}] {respond}")

        #         else:
        #             # 较早的条目：只展示工具名，不展示内容
        #             parts.append(
        #                 f"  [{tool_name}] (内容已折叠，可调用 "
        #                 f"query_tool_respond 查看)"
        #             )